# Chapter 9 — Real-Time Webcam Capture
**Project:** PTVNDM  
**Notebook:** `notebooks/01_webcam_capture.ipynb`

This notebook captures live video from the default webcam and saves it to `../outputs/captured_video.avi`.

**Controls:**
- Press **`q`** in the *'Video Capture'* window to stop recording and release all resources.

## 1 · Imports

In [ ]:
import cv2
import os

from IPython.display   import display, Javascript, Image
from google.colab.output import eval_js
from base64            import b64decode
import matplotlib.pyplot as plt
import numpy as np

print(f"OpenCV version: {cv2.__version__}")

## 2 · Configuration

In [ ]:
# ── Output settings ────────────────────────────────────────────────────────────
OUTPUT_DIR  = "/content/drive/MyDrive/PTVNDM/outputs"
OUTPUT_FILE = "captured_video.avi"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

# ── Video writer settings ──────────────────────────────────────────────────────
FOURCC      = cv2.VideoWriter_fourcc(*"XVID")  # XVID codec → .avi
FPS         = 20.0
RESOLUTION  = (640, 480)            # (width, height)

NUM_FRAMES  = 100          
CAPTURE_FPS = 20           

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output will be saved to: {os.path.abspath(OUTPUT_PATH)}")

## 3 · Capture & Record

> **Stop recording:** press **`q`** in the *'Video Capture'* window.

In [ ]:
# ── Hàm chụp 1 frame từ webcam qua JavaScript ─────────────────────────────────
# Colab chạy trên server Google nên không có webcam vật lý.
# Giải pháp: dùng JS để truy cập webcam của máy tính người dùng qua trình duyệt,
# chuyển frame thành base64 rồi gửi về Python để xử lý.

def capture_frame_js():
    js_code = Javascript("""
        async function captureFrame() {
            const video   = document.createElement('video');
            const stream  = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            const canvas  = document.createElement('canvas');
            canvas.width  = 640;
            canvas.height = 480;
            canvas.getContext('2d').drawImage(video, 0, 0, 640, 480);

            stream.getVideoTracks()[0].stop();   // tắt webcam sau khi chụp
            return canvas.toDataURL('image/jpeg', 0.8);
        }
    """)
    display(js_code)
    data   = eval_js("captureFrame()")
    binary = b64decode(data.split(",")[1])

    # Chuyển bytes → numpy array → ảnh BGR (định dạng của OpenCV)
    arr   = np.frombuffer(binary, dtype=np.uint8)
    frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    return frame

# ── 1. Tạo VideoWriter ─────────────────────────────────────────────────────────
out = cv2.VideoWriter(OUTPUT_PATH, FOURCC, FPS, RESOLUTION)

if not out.isOpened():
    raise IOError(f"Không thể tạo VideoWriter tại '{OUTPUT_PATH}'.")

print(f"Bắt đầu quay {NUM_FRAMES} frames → {os.path.abspath(OUTPUT_PATH)}")

# ── 2. Capture loop ────────────────────────────────────────────────────────────
# Thay vì dùng phím 'q' để dừng, ta giới hạn số frame vì không có GUI window.
try:
    for i in range(NUM_FRAMES):
        frame = capture_frame_js()

        if frame is None:
            print(f"  Cảnh báo: không đọc được frame {i+1}. Bỏ qua.")
            continue

        frame_resized = cv2.resize(frame, RESOLUTION)
        out.write(frame_resized)

        if (i + 1) % 10 == 0:
            print(f"  Đã quay: {i+1}/{NUM_FRAMES} frames")

finally:
    # ── 3. Giải phóng tài nguyên ───────────────────────────────────────────────
    out.release()
    print("VideoWriter đã giải phóng.")
    print(f"Video lưu tại: {os.path.abspath(OUTPUT_PATH)}")

## 4 · Verify Output

In [ ]:
if os.path.exists(OUTPUT_PATH):
    size_mb = os.path.getsize(OUTPUT_PATH) / (1024 ** 2)
    print(f"✔ File found: {os.path.abspath(OUTPUT_PATH)}")
    print(f"  Size: {size_mb:.2f} MB")

    verify2 = cv2.VideoCapture(OUTPUT_PATH)
    ret, preview = verify2.read()
    if ret:
        plt.figure(figsize=(6, 4))
        plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
        plt.title("Preview — Frame đầu tiên của video đã lưu")
        plt.axis("off")
        plt.show()
    else:
        print("Warning: File exists but could not be opened by OpenCV for verification.")
        
    verify2.release()

else:
    print(f"✘ File not found at: {os.path.abspath(OUTPUT_PATH)}")